This script links digital objects to archival objects, one-to-one. You will need to upload a CSV file with two columns containing the archival_object_uri (Column 1) and the corresponding digital object uri (Column 2) to be linked. 

# 1. Import Modules

In [ ]:
import csv
import json
import requests
import yaml
from datetime import datetime

# 2. Authenticate to the API

In [ ]:
#Get the current date. This will be used to name the files created.
report_date = datetime.now().strftime("%m_%d_%Y")

#Safe loads the secrets file containing your username and password.
with open("../secrets.yml") as f:
    secrets = yaml.safe_load(f)

#Credentials to post for authentication
baseURL = 'https://archives.pratt.edu/staff/api'
user = secrets['username']
password = secrets['password']
repository = "2" #The Pratt Archives has only one repository which will always be "2"

#Sends the authentication request to the API
auth = requests.post(baseURL + '/users/' + user + '/login?password='+ password).json()

#If authentication fails, an error will be printed.
if 'session' not in auth:
    print("Error: authentication failed.")
    print(auth)
else:
    session = auth['session']
    headers = {'X-ArchivesSpace-Session': session,
            'Content_Type': 'application/json'}

    print('Authentication successful.')

# 3. Add full file path here for CSV file with object uris to be linked. 

In [ ]:
input_csv = "C:/Users/adela448/Desktop/API_Scripts/input/digital_objects_to_link_08_06_2026.csv"

# 4. Get Archival Object and Link Digital Object

In [ ]:
with open(input_csv, 'r', encoding='utf-8') as csv_file: 
    csv_reader = csv.reader(csv_file)
    header = next(csv_reader)

    for row in csv_reader:

        archival_object_uri = row[0]
        digital_object_uri = row[1]

        #Get record for Archival Object
        output = requests.get(baseURL + archival_object_uri, headers=headers).json()

        #Populate new instance record
        new_instance = {
            "instance_type": "digital_object",
            "jsonmodel_type": "instance",
            "is_representative": False,
            "digital_object": {
                "ref": digital_object_uri
            }
        }

        output['instances'].append(new_instance)

        print(f"Updating digital object for {output.get('display_string')}")

        #Post the update
        try: 
            response = requests.post(
                baseURL + archival_object_uri,
                headers=headers,
                json=output
            )

            if response.status_code == 200:
                data = response.json()
                print(f"Updated.")
            else:
                print(f"Failed.")
                print(response.text)

        except Exception as e:
            print(f"Error updating {output['title']}.")

        print("-------------")

    print("Done.")

# 5. Publish Digital Objects (if currently unpublished)

In [ ]:
with open(input_csv, 'r', encoding='utf-8') as csv_file: 
    csv_reader = csv.reader(csv_file)
    header = next(csv_reader)

    for row in csv_reader:
        digital_object_uri = row[1]

        print(f"Publishing {digital_object_uri}...")

        try: 
            response = requests.post(
                baseURL + digital_object_uri + '/publish',
                headers=headers,
                json=output
            )

            if response.status_code == 200:
                data = response.json()
                print(f"Published.")
            else:
                print(f"Failed.")
                print(response.text)

        except Exception as e:
            print(f"Error publishing.")

        print("=======")

    print("Done.")

        